In [1]:
# 01. IMPORT VÀ CẤU HÌNH
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import random
import copy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Ưu tiên thư mục của file .py; trong Notebook dùng thư mục làm việc.
PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
# Chỉnh đường dẫn nếu file Bước 03 được lưu ở chỗ khác.
INPUT_CANDIDATES = [
    PROJECT_DIR / "03_model_ready_development.csv",
    PROJECT_DIR / "outputs/03_feature_engineering/03_model_ready_development.csv",
    Path("03_model_ready_development.csv"),
    Path("outputs/03_feature_engineering/03_model_ready_development.csv"),
]
INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.is_file()), INPUT_CANDIDATES[0])
OUTPUT_DIR = PROJECT_DIR / "outputs/05_model_training_A"

FEATURES = ["z_gap", "z_intraday", "z_range", "z_volume"]
SEED = 42
IF_TREES = 200
IF_MAX_SAMPLES = 256
LOF_NEIGHBORS = 20
AE_HIDDEN = 2
AE_LR = 0.001
AE_BATCH_SIZE = 512  # giảm số bước cập nhật mỗi epoch; vẫn huấn luyện bằng Train
AE_MAX_EPOCHS = 1000  # 300 epoch ban đầu chưa ổn định; mở rộng trong phương án A
AE_PATIENCE = 30
AE_MIN_DELTA = 1e-5

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy file Bước 03: {INPUT_PATH.resolve()}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chỉ sử dụng CPU để dễ tái lập trên máy cá nhân.
DEVICE = torch.device("cpu")
torch.set_num_threads(1)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

print("Nguồn:", INPUT_PATH.resolve())
print("Đầu ra:", OUTPUT_DIR.resolve())
print("PyTorch:", torch.__version__, "| scikit-learn:", sklearn.__version__)

Matplotlib is building the font cache; this may take a moment.


Nguồn: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\03_feature_engineering\03_model_ready_development.csv
Đầu ra: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\05_model_training_A
PyTorch: 2.15.0+cpu | scikit-learn: 1.9.1


In [2]:
# 02. ĐỌC VÀ KIỂM TRA NHÃN, THỜI GIAN, ĐẶC TRƯNG
source = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
required = ["ticker", "date", "split", "val_role", *FEATURES]
if not set(required).issubset(source.columns):
    raise ValueError(f"Thiếu cột: {sorted(set(required) - set(source.columns))}")
if source[required].isna().any().any():
    raise ValueError("Có giá trị thiếu trong khóa/nhãn/đặc trưng")
if source.duplicated(["ticker", "date"]).any():
    raise ValueError("Trùng ticker-date")
if not source["split"].isin(["train", "val"]).all():
    raise ValueError("File Development có nhãn Test hoặc nhãn lạ")
if not source.loc[source["split"].eq("val"), "val_role"].isin(["select", "calibrate"]).all():
    raise ValueError("Validation phải có val_role=select hoặc calibrate")
if not source.loc[source["split"].eq("train"), "val_role"].eq("not_applicable").all():
    raise ValueError("Train bị gắn sai val_role")

source["date"] = pd.to_datetime(source["date"], format="%Y-%m-%d", errors="raise")
for name in FEATURES:
    source[name] = pd.to_numeric(source[name], errors="raise")
if not np.isfinite(source[FEATURES].to_numpy(dtype=float)).all():
    raise ValueError("Đặc trưng không hữu hạn")

# Không dùng phần calibrate trong các phép tính tiếp theo.
train = source.loc[source["split"].eq("train"), required].copy()
val_select = source.loc[
    source["split"].eq("val") & source["val_role"].eq("select"), required
].copy()
del source

train = train.sort_values(["date", "ticker"]).reset_index(drop=True)
val_select = val_select.sort_values(["date", "ticker"]).reset_index(drop=True)
if train.empty or val_select.empty:
    raise ValueError("Train hoặc Validation select rỗng")
if train["date"].max() >= val_select["date"].min():
    raise ValueError("Rò rỉ: Train không đứng trước Validation select")
if set(train["ticker"]) != set(val_select["ticker"]):
    raise ValueError("Các mã giữa Train và Validation select không khớp")
if not (len(train) > IF_MAX_SAMPLES and len(train) > LOF_NEIGHBORS):
    raise ValueError("Train quá ít mẫu so với thông số mô hình")

X_train = train[FEATURES].to_numpy(dtype=np.float32)
X_val = val_select[FEATURES].to_numpy(dtype=np.float32)
print("Train:", len(train), "| Val select:", len(val_select), "| Mã:", train["ticker"].nunique())
print("Train kết thúc:", train["date"].max().date(), "| Val select bắt đầu:", val_select["date"].min().date())
print("Ghi chú: không có Test hoặc Validation calibrate trong X_train/X_val")

Train: 10387 | Val select: 1430 | Mã: 13
Train kết thúc: 2025-04-01 | Val select bắt đầu: 2025-04-02
Ghi chú: không có Test hoặc Validation calibrate trong X_train/X_val


In [3]:
# 03. ISOLATION FOREST
if_model = IsolationForest(
    n_estimators=IF_TREES,
    max_samples=IF_MAX_SAMPLES,
    max_features=1.0,
    bootstrap=False,
    contamination="auto",
    random_state=SEED,
    n_jobs=1,
)
if_model.fit(X_train)
if_train_scores = -if_model.score_samples(X_train)
if_val_scores = -if_model.score_samples(X_val)
assert np.isfinite(if_train_scores).all() and np.isfinite(if_val_scores).all()
print("IF đã fit bằng Train; chấm Val select thành công")

IF đã fit bằng Train; chấm Val select thành công


In [4]:
# 04. LOCAL OUTLIER FACTOR
lof_model = LocalOutlierFactor(
    n_neighbors=LOF_NEIGHBORS,
    metric="minkowski",
    p=2,
    novelty=True,
    contamination="auto",
    n_jobs=1,
)
lof_model.fit(X_train)
lof_val_scores = -lof_model.score_samples(X_val)
assert np.isfinite(lof_val_scores).all()
print("LOF đã fit bằng Train; chấm Val select (novelty) thành công")

LOF đã fit bằng Train; chấm Val select (novelty) thành công


In [5]:
# 05. AUTOENCODER VÀ VÒNG HUẤN LUYỆN
class SmallAutoencoder(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),  # Output tuyến tính, nhận cả số âm/dương
        )

    def forward(self, x):
        return self.net(x)


ae_model = SmallAutoencoder(input_dim=len(FEATURES), hidden_dim=AE_HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=AE_LR)
loss_fn = nn.MSELoss(reduction="mean")

train_tensor = torch.from_numpy(X_train).to(DEVICE)
val_tensor = torch.from_numpy(X_val).to(DEVICE)
loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=AE_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)

best_state = None
best_val_mse = float("inf")
best_epoch = None
no_improve = 0
history = []

for epoch in range(1, AE_MAX_EPOCHS + 1):
    ae_model.train()
    for (batch,) in loader:
        optimizer.zero_grad(set_to_none=True)
        reconstruction = ae_model(batch)
        loss = loss_fn(reconstruction, batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Loss không hữu hạn tại epoch {epoch}")
        loss.backward()
        optimizer.step()

    ae_model.eval()
    with torch.no_grad():
        train_mse = loss_fn(ae_model(train_tensor), train_tensor).item()
        val_mse = loss_fn(ae_model(val_tensor), val_tensor).item()
    if not np.isfinite(train_mse) or not np.isfinite(val_mse):
        raise RuntimeError("MSE không hữu hạn: cần kiểm tra cách huấn luyện AE")

    history.append({"epoch": epoch, "train_mse": train_mse, "val_select_mse": val_mse})
    if val_mse < best_val_mse - AE_MIN_DELTA:
        best_val_mse = val_mse
        best_epoch = epoch
        best_state = copy.deepcopy(ae_model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if epoch == 1 or epoch % 100 == 0:
        print(f"AE epoch {epoch:3d}: train MSE={train_mse:.4f} | val-select MSE={val_mse:.4f}")
    if no_improve >= AE_PATIENCE:
        print(f"Dừng sớm AE ở epoch {epoch} (patience={AE_PATIENCE})")
        break

if best_state is None:
    raise RuntimeError("Không có epoch hợp lệ cho AE")
ae_model.load_state_dict(best_state)
ae_model.eval()

with torch.no_grad():
    recon_train = ae_model(train_tensor).cpu().numpy()
    recon_val = ae_model(val_tensor).cpu().numpy()
ae_train_errors = (X_train - recon_train) ** 2
ae_val_errors = (X_val - recon_val) ** 2
ae_train_scores = ae_train_errors.mean(axis=1)
ae_val_scores = ae_val_errors.mean(axis=1)
assert np.isfinite(ae_train_scores).all() and np.isfinite(ae_val_scores).all()
print("AE best epoch:", best_epoch, "| best Val-select MSE:", round(best_val_mse, 6))

C:\Users\SON\AppData\Local\Temp\ipykernel_25172\687093416.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  train_tensor = torch.from_numpy(X_train).to(DEVICE)


AE epoch   1: train MSE=4.7046 | val-select MSE=7.7256
AE epoch 100: train MSE=3.3425 | val-select MSE=4.9659
AE epoch 200: train MSE=3.0238 | val-select MSE=3.7868
AE epoch 300: train MSE=2.8295 | val-select MSE=3.0026
AE epoch 400: train MSE=2.6728 | val-select MSE=2.4401
AE epoch 500: train MSE=2.5416 | val-select MSE=2.0504
AE epoch 600: train MSE=2.4262 | val-select MSE=1.7658
AE epoch 700: train MSE=2.3236 | val-select MSE=1.5590
AE epoch 800: train MSE=2.2328 | val-select MSE=1.4211
AE epoch 900: train MSE=2.1488 | val-select MSE=1.3096
AE epoch 1000: train MSE=2.0743 | val-select MSE=1.2361
AE best epoch: 997 | best Val-select MSE: 1.226357


In [6]:
# 06. LƯU ĐIỂM MÔ HÌNH VÀ PHÂN TÍCH AE
val_scores = val_select[["ticker", "date", "split", "val_role"]].copy()
val_scores["score_if"] = if_val_scores
val_scores["score_ae"] = ae_val_scores
val_scores["score_lof"] = lof_val_scores
val_scores["date"] = val_scores["date"].dt.strftime("%Y-%m-%d")
val_scores.to_csv(OUTPUT_DIR / "05_val_select_scores.csv", index=False, encoding="utf-8-sig")

train_scores = train[["ticker", "date", "split"]].copy()
train_scores["score_if"] = if_train_scores
train_scores["score_ae"] = ae_train_scores
train_scores["date"] = train_scores["date"].dt.strftime("%Y-%m-%d")
train_scores.to_csv(OUTPUT_DIR / "05_train_scores_diagnostics.csv", index=False, encoding="utf-8-sig")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "05_ae_training_history.csv", index=False)

rows = []
for name, err in [("train", ae_train_errors), ("val_select", ae_val_errors)]:
    total = float(err.sum())
    for j, f in enumerate(FEATURES):
        rows.append({
            "subset": name,
            "feature": f,
            "mse": float(err[:, j].mean()),
            "mae": float(np.sqrt(err[:, j]).mean()),
            "share_squared_error_pct": float(100 * err[:, j].sum() / total),
        })
ae_error_summary = pd.DataFrame(rows)
ae_error_summary.to_csv(OUTPUT_DIR / "05_ae_error_by_feature.csv", index=False)

# Chẩn đoán phân bố điểm theo mã, không gắn ngưỡng.
score_rows = []
for name, g in val_scores.groupby("ticker", sort=True):
    for m in ["score_if", "score_ae", "score_lof"]:
        v = g[m].to_numpy(dtype=float)
        score_rows.append({
            "ticker": name, "model": m, "n": len(v),
            "median_score": float(np.median(v)),
            "q90_score": float(np.quantile(v, 0.90)),
            "q99_score": float(np.quantile(v, 0.99)),
            "max_score": float(np.max(v)),
        })
pd.DataFrame(score_rows).to_csv(
    OUTPUT_DIR / "05_val_select_score_distribution_by_ticker.csv", index=False
)
print("\nAE — tỷ trọng lỗi tái tạo theo thành phần (%):")
print(ae_error_summary.pivot(index="feature", columns="subset", values="share_squared_error_pct").round(2))


AE — tỷ trọng lỗi tái tạo theo thành phần (%):
subset      train  val_select
feature                      
z_gap       69.70       29.76
z_intraday  19.04       47.70
z_range      4.22       10.73
z_volume     7.03       11.81


In [7]:
# 07. LƯU MÔ HÌNH VÀ KIỂM TRA RELOAD
joblib.dump(if_model, OUTPUT_DIR / "05_if.joblib")
joblib.dump(lof_model, OUTPUT_DIR / "05_lof.joblib")
torch.save(
    {"state_dict": ae_model.state_dict(), "input_dim": len(FEATURES), "hidden_dim": AE_HIDDEN},
    OUTPUT_DIR / "05_ae.pt",
)

# Không tải checkpoint từ nguồn không tin cậy. Test reload mô hình mới lưu.
assert np.allclose(-joblib.load(OUTPUT_DIR / "05_if.joblib").score_samples(X_val), if_val_scores)
assert np.allclose(-joblib.load(OUTPUT_DIR / "05_lof.joblib").score_samples(X_val), lof_val_scores)
checkpt = torch.load(OUTPUT_DIR / "05_ae.pt", map_location="cpu", weights_only=True)
ae_reloaded = SmallAutoencoder(checkpt["input_dim"], checkpt["hidden_dim"])
ae_reloaded.load_state_dict(checkpt["state_dict"])
ae_reloaded.eval()
with torch.no_grad():
    ae_reload_score = ((val_tensor - ae_reloaded(val_tensor)) ** 2).mean(dim=1).numpy()
assert np.allclose(ae_reload_score, ae_val_scores, atol=1e-5)

# Chấm cùng một dòng theo thứ tự khác nhau phải cho cùng điểm.
check_idx = np.array([0, len(X_val) // 2, len(X_val) - 1])
assert np.allclose(-if_model.score_samples(X_val[check_idx][::-1])[::-1], if_val_scores[check_idx])
assert np.allclose(-lof_model.score_samples(X_val[check_idx][::-1])[::-1], lof_val_scores[check_idx])

sha = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
run_config = {
    "experiment": "EX1_baseline_A_step05",
    "input_path": str(INPUT_PATH.resolve()),
    "input_sha256": sha,
    "features": FEATURES,
    "normalization": "Step03 rolling median/MAD W=60; no second scaler",
    "train_rows": len(train),
    "val_select_rows": len(val_select),
    "train_max_date": str(train["date"].max().date()),
    "val_select_max_date": str(val_select["date"].max().date()),
    "random_seed": SEED,
    "if": {"n_estimators": IF_TREES, "max_samples": IF_MAX_SAMPLES, "max_features": 1.0, "contamination": "auto"},
    "lof": {"n_neighbors": LOF_NEIGHBORS, "metric": "minkowski", "p": 2, "novelty": True, "contamination": "auto"},
    "ae": {
        "architecture": "4-tanh(2)-4_linear", "loss": "MSE", "optimizer": "Adam", "lr": AE_LR,
        "batch_size": AE_BATCH_SIZE, "max_epochs": AE_MAX_EPOCHS, "early_stopping_patience": AE_PATIENCE,
        "early_stopping_min_delta": AE_MIN_DELTA, "best_epoch": int(best_epoch),
        "epochs_executed": len(history), "best_val_select_mse": float(best_val_mse),
        "early_stopping_triggered": len(history) < AE_MAX_EPOCHS,
        "warning": "AE reached max epochs before early stopping" if len(history) == AE_MAX_EPOCHS else None,
    },
    "policies": ["no Test or Val-calibrate used", "no alert thresholds in Step05", "no gap-row removal", "LOF train negative_outlier_factor not compared to novelty score"],
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__},
}
(OUTPUT_DIR / "05_run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")

1583

In [8]:
# 08. BIỂU ĐỒ VÀ BÁO CÁO
history_df = pd.DataFrame(history)
plt.figure(figsize=(8, 4.5))
plt.plot(history_df["epoch"], history_df["train_mse"], label="Train MSE")
plt.plot(history_df["epoch"], history_df["val_select_mse"], label="Val-select MSE")
plt.axvline(best_epoch, linestyle="--", label=f"Best epoch {best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("MSE (thang đặc trưng baseline A)")
plt.title("Autoencoder — quá trình huấn luyện baseline A")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_ae_learning_curve.png", dpi=150)
plt.close()

train_gap_mask = np.abs(X_train[:, 0]) > 30
train_gap_count = int(train_gap_mask.sum())
train_gap_share = float(100 * (X_train[train_gap_mask] ** 2).sum() / (X_train ** 2).sum())
val_reconstruction_gap_pct = float(ae_error_summary.loc[
    (ae_error_summary.subset == "val_select") & (ae_error_summary.feature == "z_gap"),
    "share_squared_error_pct"
].iloc[0])

report = f"""# Bước 05 — Model Training, phương án A

**Trạng thái:** đã huấn luyện IF, AE, LOF bằng Train; chấm điểm Validation select. **Chưa hiệu chỉnh ngưỡng và chưa đánh giá Test.**

## Dữ liệu
- Train: **{len(train):,}** phiên thuộc {train['ticker'].nunique()} mã.
- Validation select: **{len(val_select):,}** phiên.
- Train kết thúc: {train['date'].max().date()}; Val-select bắt đầu: {val_select['date'].min().date()}.
- Đầu vào: {', '.join(FEATURES)}; không thêm chuẩn hóa lần hai, không loại gap cực trị.

## Mô hình
- IF: {IF_TREES} cây, {IF_MAX_SAMPLES} mẫu/cây, `contamination='auto'` nhưng chấm bằng `-score_samples`.
- LOF: {LOF_NEIGHBORS} hàng xóm, Euclid, `novelty=True`; chấm Val-select bằng `-score_samples`.
- AE: 4→2→4, tanh, MSE, Adam lr={AE_LR}; chạy {len(history)} epoch; checkpoint tốt nhất epoch **{best_epoch}** với Val-select MSE **{best_val_mse:.6f}**.
- **Cảnh báo:** AE {"đã dừng sớm" if len(history) < AE_MAX_EPOCHS else "chạm giới hạn số epoch; dừng sớm chưa xảy ra"}. Chưa được coi là đã hội tụ hoặc cấu hình tối ưu. Kiểm tra ngân sách huấn luyện tiếp trong Bước 06.

## Chẩn đoán (chưa phải đánh giá độ chính xác)
- Train có **{train_gap_count}** phiên `|z_gap| > 30`, chiếm **{train_gap_share:.2f}%** tổng bình phương của cả bốn đặc trưng.
- Trong Validation select, thành phần gap chiếm **{val_reconstruction_gap_pct:.2f}%** tổng bình phương lỗi tái tạo AE.
- Cần xem chi tiết lỗi AE theo từng thành phần trong `05_ae_error_by_feature.csv` và bảng điểm theo mã.
- Các kết quả trên **không chứng minh phiên nào là bất thường thật hoặc mô hình nào tốt nhất**.

## Kiểm tra đã thực hiện
- Train và Validation select tách đúng thứ tự thời gian.
- Cùng bốn đặc trưng cho ba mô hình; không fit `calibrate` hoặc Test.
- Các điểm IF/AE/LOF hữu hạn, lưu/tải lại khớp.
- Không tạo nhãn, không tạo ngưỡng, không tính Accuracy/Precision/Recall/F1 từ proxy.
- **Lưu ý:** LOF novelty chỉ dùng điểm trên dữ liệu mới; không so trực tiếp với `negative_outlier_factor_` trên Train.

## Bước tiếp theo
Bước 06: Thử sự ổn định của IF/AE, số hàng xóm LOF, ảnh hưởng các phiên gap cực trị và dạng biến động OHLCV trên Train + Validation select. Chỉ cân nhắc phương án B sau khi đánh giá đầy đủ phương án A.
"""
(OUTPUT_DIR / "05_training_report.md").write_text(report, encoding="utf-8")
print("\nĐÃ HOÀN THÀNH BƯỚC 05 — BASELINE A")
print("Best epoch AE:", best_epoch)
if len(history) == AE_MAX_EPOCHS:
    print("CẢNH BÁO: AE chạm tối đa epoch khi Val-select MSE vẫn còn giảm; cần kiểm tra ở Bước 06")
print("Val select scores:", val_scores.shape)
print("Báo cáo:", OUTPUT_DIR / "05_training_report.md")
print("Không dùng Validation calibrate hoặc Test; không có ngưỡng hay nhãn bất thường.")


ĐÃ HOÀN THÀNH BƯỚC 05 — BASELINE A
Best epoch AE: 997
CẢNH BÁO: AE chạm tối đa epoch khi Val-select MSE vẫn còn giảm; cần kiểm tra ở Bước 06
Val select scores: (1430, 7)
Báo cáo: d:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\05_model_training_A\05_training_report.md
Không dùng Validation calibrate hoặc Test; không có ngưỡng hay nhãn bất thường.
